In [3]:
# =============================
# 1️⃣ Imports
# =============================

import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# =============================
# 2️⃣ Carregar base (ajuste caminho se precisar)
# =============================

df_public = pd.read_csv('C:/Users/Doit/Desktop/data-science-challenge-at-eef-2024/spaceflights/data/01_raw/public.csv')

# =============================
# 3️⃣ Engenharia de data/hora
# =============================

df_public['hora_ref'] = pd.to_datetime(df_public['hora_ref'])
df_public['hora'] = df_public['hora_ref'].dt.hour
df_public['dia'] = df_public['hora_ref'].dt.dayofyear

# =============================
# 4️⃣ Feature de troca
# =============================

df_public['precisa_troca'] = df_public['prev_troca_cabeceira'] | df_public['troca_cabeceira_hora_anterior']

# =============================
# 5️⃣ Decodificação METAR
# =============================

def parse_metar(metar):
    data = {}
    if pd.isnull(metar):
        return {
            'wind_dir': 0,
            'wind_speed': 0,
            'visibility': 0,
            'few_clouds': 0,
            'sct_clouds': 0,
            'bkn_clouds': 0,
            'ovc_clouds': 0,
            'temp': 0,
            'dew_point': 0,
            'pressure': 0
        }

    wind = re.search(r'(\d{3})(\d{2})KT', metar)
    data['wind_dir'] = int(wind.group(1)) if wind else 0
    data['wind_speed'] = int(wind.group(2)) if wind else 0

    vis = re.search(r'\s(\d{4})\s', metar)
    data['visibility'] = int(vis.group(1)) if vis else 0

    data['few_clouds'] = int(bool(re.search(r'FEW\d{3}', metar)))
    data['sct_clouds'] = int(bool(re.search(r'SCT\d{3}', metar)))
    data['bkn_clouds'] = int(bool(re.search(r'BKN\d{3}', metar)))
    data['ovc_clouds'] = int(bool(re.search(r'OVC\d{3}', metar)))

    temp = re.search(r'(\d{2})/(\d{2})', metar)
    data['temp'] = int(temp.group(1)) if temp else 0
    data['dew_point'] = int(temp.group(2)) if temp else 0

    q = re.search(r'Q(\d{4})', metar)
    data['pressure'] = int(q.group(1)) if q else 0

    return data

metar_features = df_public['metar'].apply(parse_metar)
metar_df = pd.DataFrame(list(metar_features))
df_public = pd.concat([df_public, metar_df], axis=1)

# =============================
# 6️⃣ Decodificação METAF (exemplo simplificado)
# =============================

def parse_metaf(metaf):
    data = {}
    if pd.isnull(metaf):
        return {'forecast_wind_speed': 0, 'forecast_visibility': 0}

    wind = re.search(r'(\d{3})(\d{2})KT', metaf)
    data['forecast_wind_speed'] = int(wind.group(2)) if wind else 0

    vis = re.search(r'\s(\d{4})\s', metaf)
    data['forecast_visibility'] = int(vis.group(1)) if vis else 0

    return data

metaf_features = df_public['metaf'].apply(parse_metaf)
metaf_df = pd.DataFrame(list(metaf_features))
df_public = pd.concat([df_public, metaf_df], axis=1)

# =============================
# 7️⃣ One-Hot Encoding de origem/destino
# =============================

df_public = pd.get_dummies(df_public, columns=['origem', 'destino'], drop_first=True)

# =============================
# 8️⃣ Preparar treino e previsão
# =============================

df_train = df_public[df_public['espera'].notnull()]
df_pred = df_public[df_public['espera'].isnull()]

# Remova colunas que não são features úteis
X_train = df_train.drop(columns=[
    'espera', 'flightid', 'hora_ref', 'url_img_satelite', 'metar', 'metaf'
])
y_train = df_train['espera']

X_to_predict = df_pred.drop(columns=[
    'espera', 'flightid', 'hora_ref', 'url_img_satelite', 'metar', 'metaf'
])

# Alinhar colunas
X_train, X_to_predict = X_train.align(X_to_predict, join='left', axis=1, fill_value=0)

# =============================
# 9️⃣ SMOTE e treino
# =============================

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

sm = SMOTE(random_state=42)
X_tr_res, y_tr_res = sm.fit_resample(X_tr, y_tr)

clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_tr_res, y_tr_res)

y_val_pred = clf.predict(X_val)
print(confusion_matrix(y_val, y_val_pred))
print(classification_report(y_val, y_val_pred))

# =============================
# 🔟 Previsão final para os NaN
# =============================

espera_pred = clf.predict(X_to_predict)
df_pred['espera_pred'] = espera_pred

print(df_pred['espera_pred'].value_counts())

# =============================
# ✅ Submissão
# =============================

df_submit = df_pred[['flightid', 'espera_pred']]
df_submit.to_csv('submission.csv', index=False)


[[41165   434]
 [  622   115]]
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     41599
         1.0       0.21      0.16      0.18       737

    accuracy                           0.98     42336
   macro avg       0.60      0.57      0.58     42336
weighted avg       0.97      0.98      0.97     42336

espera_pred
0.0    90344
1.0      376
Name: count, dtype: int64


C:\Users\Doit\AppData\Local\Temp\ipykernel_26932\4198693897.py:148: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pred['espera_pred'] = espera_pred


In [6]:
df_submission = pd.read_csv("submission.csv")

In [7]:
df_submission.shape

(90720, 2)